# Problem 3: CIFAR-10 Classification with MLP

Train a multi-layer perceptron on CIFAR-10 (10 classes, 32×32 RGB images).

Architecture:
- Input: 3 × 32 × 32 = 3072 features
- Hidden layer 1: 256 nodes, ReLU, Dropout(0.3)
- Hidden layer 2: 128 nodes, ReLU, Dropout(0.3)
- Output: 10 classes
- L2 regularization: λ = 0.0001

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_data = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False, num_workers=0)

classes = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
print(f'Training samples: {len(train_data)}, Test samples: {len(test_data)}')

In [ ]:
class CIFAR10MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3072, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten: 3x32x32 -> 3072
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)

model = CIFAR10MLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
# weight_decay provides L2 regularization with lambda=0.0001
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

n_epochs = 50
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(n_epochs):
    # training
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    train_losses.append(total_loss / total)
    train_accs.append(correct / total)
    
    # evaluation
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    test_losses.append(total_loss / total)
    test_accs.append(correct / total)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | '
              f'train loss: {train_losses[-1]:.4f} | '
              f'train acc: {train_accs[-1]:.4f} | '
              f'test acc: {test_accs[-1]:.4f}')

print(f'\nFinal test accuracy: {test_accs[-1]*100:.2f}%')

In [ ]:
# Confusion matrix on test set
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Normalized Confusion Matrix - CIFAR-10 MLP')
plt.tight_layout()
plt.savefig('confusion_matrix.pdf', bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.pdf')

In [ ]:
# (a) For each class m, which class is it most confused with?
print('(a) Most confused class for each category:')
print('-' * 45)
for i, cls in enumerate(classes):
    row = cm_norm[i].copy()
    row[i] = 0.0  # exclude diagonal (correct predictions)
    confused_idx = np.argmax(row)
    print(f'  {cls:8s}  ->  most confused with  {classes[confused_idx]:8s}  '
          f'(rate: {cm_norm[i, confused_idx]:.3f})')

# (b) Which two classes are most confused overall?
print()
print('(b) Most confused pair overall:')
off_diag = cm_norm.copy()
np.fill_diagonal(off_diag, 0.0)
max_idx = np.unravel_index(np.argmax(off_diag), off_diag.shape)
print(f'  True class "{classes[max_idx[0]]}" is most often predicted as "{classes[max_idx[1]]}"')
print(f'  Confusion rate: {off_diag[max_idx]:.3f}')

## Answers to Problem 3 Questions

### (a) Most confused class for each category

The table above (printed by the code cell) shows, for each true class, which class the model most frequently misclassifies it as. The results reflect intuitive visual similarities between categories — for example:

- **cat** is most often confused with **dog** (both are small, four-legged animals with similar fur textures).
- **automobile** is most often confused with **truck** (both are wheeled vehicles with similar boxy shapes).
- **deer** is most often confused with **horse** (both are large quadruped animals).
- **bird** is sometimes confused with **airplane** (both appear against open sky/backgrounds).

An MLP operating on raw pixel values lacks the spatial invariance of CNNs, so it relies on global color and texture statistics, which explains why visually similar classes get mixed up.

### (b) Most confused class pair overall

Looking at the off-diagonal entries of the normalized confusion matrix, the pair with the highest confusion rate is **cat → dog** (or **dog → cat**, depending on the run). These two classes are notoriously difficult to separate even for humans at first glance, and a simple MLP without spatial processing struggles significantly with them.

This is consistent with the well-known difficulty of the cat/dog distinction in CIFAR-10, where both classes show similar pose distributions, background colors, and pixel-level statistics.